# v2 analysis

Every figure and table the spec (`docs/spec.md`) lists, read from the results command's saved files. Nothing here fits a model.

Each reading's outcome is computed from the pre-registered rules in `backtest/analysis.py`, not written by hand. Statistics use the seed-averaged series, at the settings recorded in the run's `run.json`.

`RESULTS` below is the only line that changes between the rehearsal outputs and the official ones.

In [ ]:
RESULTS = "results-rehearsal"  # the official run: "results"

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
sys.path.insert(0, str(ROOT))
from backtest import analysis as a
from backtest.results import ARMS, PRIMARY_PAIR
from backtest.stats import exposure_matched_benchmark

RESULTS = ROOT / RESULTS
series, trades, windows, settings = a.load(RESULTS)
run_info = json.loads((RESULTS / "run.json").read_text())
{k: run_info[k] for k in ["official_run", "git", "rows", "first_date", "last_date", "prices_sha256"]}

In [ ]:
CI = f"{1 - settings.ci_alpha:.0%} CI"
LABELS = {"v1_style": "headline arm: v1-style arm (v1's features inside the corrected pipeline)", "stationary_twin": "stationary twin"}
bench = a.benchmark(series)


def rows(runs, round_trip_bps=None):
    return pd.DataFrame({r: a.summary(series, trades, r, settings, round_trip_bps) for r in runs}).T


def fmt(t):
    """A summary table as readable strings. Excess return is the mean daily excess, in bps."""
    return pd.DataFrame({
        "strategy return": t.strategy_return.map("{:+.1%}".format),
        f"Sharpe [{CI}]": [f"{s:.2f} [{lo:.2f}, {hi:.2f}]" for s, (lo, hi) in zip(t.sharpe, t.sharpe_ci)],
        f"excess, bps/day [{CI}]": [f"{e * 1e4:+.2f} [{lo * 1e4:+.2f}, {hi * 1e4:+.2f}]"
                                    for e, (lo, hi) in zip(t.excess, t.excess_ci)],
        "break-even, bps round trip": t.break_even.map(lambda b: "None" if b is None else f"{b:.0f}"),
        "trades per seed": t.trades_per_seed.map("{:.1f}".format),
        "per-seed Sharpe, mean (min to max)": [f"{m:.2f} ({lo:.2f} to {hi:.2f})" for m, lo, hi in
                                               zip(t.seed_sharpe_mean, t.seed_sharpe_min, t.seed_sharpe_max)],
    })

## Headline figure

Equity curves of the seed-averaged series, net of default costs. Each arm's exposure-matched benchmark is buy-and-hold scaled by that arm's whole-series average position, with no costs: dashed, in the arm's colour.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for run, color in zip(PRIMARY_PAIR, ["C0", "C1"]):
    r = a.per_seed(series, trades, run, settings).mean(axis=1)
    ax.plot((1 + r).cumprod(), color=color, label=LABELS[run])
    matched = exposure_matched_benchmark(bench, a.position(series, run))
    ax.plot((1 + matched).cumprod(), color=color, ls="--", label=f"exposure-matched benchmark, {run.replace('_', '-')}")
ax.axhline(1, color="grey", lw=0.5)
ax.set(title=f"Out-of-sample equity, seed-averaged over {len(settings.selection_seeds)} selection seeds", ylabel="growth of 1, net of costs")
ax.legend()
fig.savefig(RESULTS / "headline.png", dpi=150, bbox_inches="tight")

## Headline table and the readings

Primary pair, seed-averaged series, default costs. Risk-free rate zero.

In [ ]:
headline = rows(PRIMARY_PAIR)
fmt(headline)

In [ ]:
r1 = a.reading1(a.per_seed(series, trades, "v1_style", settings),
                a.per_seed(series, trades, "stationary_twin", settings), settings)
# Reading 2: neither arm's whole-series excess-return CI lies above zero.
r2 = not any(lo > 0 for lo, _ in headline.excess_ci)
lo, hi = r1["diff_ci"]
print(f"""Reading 1 (two-sided), v1-style minus stationary twin:
  gate 1, paired {CI} on the daily return difference: [{lo * 1e4:+.2f}, {hi * 1e4:+.2f}] bps -> {'pass' if r1['gate1'] else 'fail'}
  gate 2, gap in mean per-seed Sharpe {r1['sharpe_gap']:+.2f} against selection spread {r1['selection_spread']:.2f} -> {'pass' if r1['gate2'] else 'fail'}
  outcome: {r1['outcome']}
Reading 2 (the strategy never worked): {'holds' if r2 else 'does not hold'}""")

### B10 cross-check

Excess return recomputed with costs at each reported break-even is zero.

In [ ]:
for run in list(ARMS) + [f"{r}_meta" for r in PRIMARY_PAIR] + ["buy_and_hold", "momentum", "mean_reversion"]:
    be = a.summary(series, trades, run, settings)["break_even"]
    if be is None or not np.isfinite(be):
        print(f"{run}: break-even {be}, nothing to check")
        continue
    excess = a.summary(series, trades, run, settings, round_trip_bps=be)["excess"]
    assert abs(excess) < 1e-12, (run, be, excess)
    print(f"{run}: excess at {be:.1f} bps = {excess:.1e}")

## Per window

Compounded strategy, exposure-matched benchmark (each window's own average position) and their difference, per test window. Not tested.

Reading 3 (descriptive, no test, primary pair only): an arm beats its exposure-matched benchmark in the early windows and trails it in the late ones, measured by mean daily excess return. A window belongs to the side of the reading-3 boundary that holds most of its test days.

In [ ]:
test_windows = windows[windows.run == "v1_style"].drop_duplicates("test_start")
window_tables, reading3 = {}, {}
for run in ARMS:
    r, pos = a.per_seed(series, trades, run, settings).mean(axis=1), a.position(series, run)
    t = a.per_window(r, pos, bench, test_windows, settings)
    window_tables[run] = t[["strategy", "benchmark", "excess"]].map("{:+.1%}".format)
    if run in PRIMARY_PAIR:
        reading3[run] = a.reading3(r, pos, bench, test_windows, settings)
pd.concat({"window": t[["test_start", "test_end", "side"]], **window_tables}, axis=1)

In [ ]:
pd.DataFrame({run: {"early, bps/day": f"{d['early'] * 1e4:+.2f}", "late, bps/day": f"{d['late'] * 1e4:+.2f}",
                    "reading 3 pattern": "yes" if d["holds"] else "no"} for run, d in reading3.items()}).T

## Price only, early and late

Cost-free buy-and-hold over the out-of-sample series, split at the reading-3 boundary. This table goes in the README whatever the models show.

In [ ]:
p = a.price_only(series, settings)
p.assign(**{"return": p["return"].map("{:+.1%}".format), "volatility": p.volatility.map("{:.0%}".format),
            "sharpe": p.sharpe.map("{:.2f}".format)})

## Cost stresses

Primary pair, recomputed from gross returns and trade dates (costs never change a decision), keeping the sell-side gap.

In [ ]:
costs = [settings.buy_cost_bps + settings.sell_cost_bps] + [b + s for b, s in settings.cost_stresses]
pd.concat({f"{c:.0f} bps": fmt(rows(PRIMARY_PAIR, c)) for c in costs}).drop(columns="break-even, bps round trip")

## Meta-labelling ablation

Primary pair, each base run beside its meta-labelled run. A window whose out-of-fold predictions had no +1 call trades as the base run does (Amendment 2).

In [ ]:
fmt(rows([r for base in PRIMARY_PAIR for r in (base, f"{base}_meta")]))

In [ ]:
meta = windows[windows.run.str.endswith("_meta")]
meta.groupby("run").oof_long_calls.agg(windows="size", fallback_windows=lambda c: int((c == 0).sum()),
                                       median_oof_long_calls="median")

## Secondary pair, descriptive

Every feature, lags included, goes through clustering. No gates.

In [ ]:
fmt(rows(["clustered_level", "clustered_stationary"]))

## Baselines

Through the same trade simulator and default costs. Each is a single run, so trades per seed is its trade count.

In [ ]:
fmt(rows(["buy_and_hold", "momentum", "mean_reversion"])).drop(columns="per-seed Sharpe, mean (min to max)")